# Coverage-Guided Fuzz Test Input Generation (SuT: `calculate_shipping_fee`)

In this exercise, we'll implement a simple **coverage-guided fuzzing** algorithm that generates random test inputs and selects those that increase structural coverages.

In [1]:
import os, sys

# exercises → unit folder → modules → project_root
MODULES_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
CODE_DIR = os.path.join(MODULES_ROOT, "exercise_artifacts", "code")
MODEL_DIR = os.path.join(MODULES_ROOT, "exercise_artifacts", "model")
DATA_DIR = os.path.join(MODULES_ROOT, "exercise_artifacts", "data")

if CODE_DIR not in sys.path:
    sys.path.append(CODE_DIR)

if MODEL_DIR not in sys.path:
    sys.path.append(MODEL_DIR)

if DATA_DIR not in sys.path:
    sys.path.append(DATA_DIR)

print("Added to sys.path:", CODE_DIR)
print("Added to sys.path:", MODEL_DIR)
print("Added to sys.path:", DATA_DIR)

from shipping_fee_instrumentation import *

stmt_tracker = StatementCoverageTracker()
branch_tracker = BranchCoverageTracker()
path_tracker = PathCoverageTracker()

print("Three structural coverage trackers have been initialized.")

Added to sys.path: /workspace/modules/exercise_artifacts/code
Added to sys.path: /workspace/modules/exercise_artifacts/model
Added to sys.path: /workspace/modules/exercise_artifacts/data
Three structural coverage trackers have been initialized.


## Random Input Generator

In [2]:
# Pure random generator (baseline)
import random

def generate_random_test_input():
    order_total = random.randint(0, 120000)
    weight_kg = round(random.uniform(0, 50), 2)
    distance_km = random.randint(0, 200)
    is_island = random.choice([True, False])
    membership = random.choice(["NONE", "WOW"])
    coupon_type = random.choice(["NONE", "NEW_USER"])
    return order_total, weight_kg, distance_km, is_island, membership, coupon_type

## Statement Coverage-Guided Generation

In [4]:
BUDGET = 100

test_inputs = []

for i in range(BUDGET):
    stmt_tracker.reset()

    # Get coverage of currently selected test inputs
    for selected_test_input in test_inputs:
        stmt_tracker.run_test(*selected_test_input)
    recent_coverage, _, _ = stmt_tracker.calculate_coverage()
    
    # Generate a new random test input
    random_test_input = generate_random_test_input()

    # Run the new test input and check if coverage increases
    stmt_tracker.run_test(*random_test_input)
    new_coverage, _, _ = stmt_tracker.calculate_coverage()
    
    # Select the test input if it increases coverage
    if new_coverage > recent_coverage:
        test_inputs.append(random_test_input)

# Print final report for selected test inputs
stmt_tracker.reset()
for test_input in test_inputs:
    stmt_tracker.run_test(*test_input)
stmt_tracker.print_report()

STATEMENT COVERAGE REPORT

Overall Coverage: 100.00% (12/12 items)
Total Tests: 6

Test Cases:
1: (67279, 31.25, 24, False, 'NONE', 'NEW_USER')
2: (111773, 25.11, 99, False, 'WOW', 'NONE')
3: (43444, 2.43, 75, True, 'NONE', 'NONE')
4: (45028, 18.81, 178, True, 'NONE', 'NONE')
5: (75908, 46.57, 1, True, 'WOW', 'NEW_USER')
6: (30794, 17.97, 35, False, 'WOW', 'NONE')

| Test input             | 1    | 2    | 3    | 4    | 5    | 6    | Covered   |
|------------------------|------|------|------|------|------|------|-----------|
| fee = 0                | O    | O    | O    | O    | O    | O    | O         |
| fee += 3000            | X    | X    | X    | X    | X    | O    | O         |
| fee += 0 (weight)      | X    | X    | O    | X    | X    | X    | O         |
| fee += 2000            | X    | X    | X    | O    | X    | O    | O         |
| fee += 5000            | O    | O    | X    | X    | O    | X    | O         |
| fee += 0 (distance)    | X    | X    | X    | X    | O    | X  

## Branch Coverage-Guided Generation

In [5]:
BUDGET = 100

test_inputs = []

for i in range(BUDGET):
    branch_tracker.reset()

    # Get coverage of currently selected test inputs
    for selected_test_input in test_inputs:
        branch_tracker.run_test(*selected_test_input)
    recent_coverage, _, _ = branch_tracker.calculate_coverage()
    
    # Generate a new random test input
    random_test_input = generate_random_test_input()

    # Run the new test input and check if coverage increases
    branch_tracker.run_test(*random_test_input)
    new_coverage, _, _ = branch_tracker.calculate_coverage()
    
    # Select the test input if it increases coverage
    if new_coverage > recent_coverage:
        test_inputs.append(random_test_input)

# Print final report for selected test inputs
branch_tracker.reset()
for test_input in test_inputs:
    branch_tracker.run_test(*test_input)
branch_tracker.print_report()

BRANCH COVERAGE REPORT

Overall Coverage: 100.00% (16/16 items)
Total Tests: 6

Test Cases:
1: (105883, 29.03, 56, True, 'NONE', 'NEW_USER')
2: (119396, 7.99, 162, False, 'WOW', 'NONE')
3: (10890, 28.53, 88, True, 'NONE', 'NEW_USER')
4: (59913, 8.1, 11, True, 'NONE', 'NEW_USER')
5: (18512, 2.62, 70, False, 'WOW', 'NONE')
6: (72465, 9.64, 0, True, 'WOW', 'NEW_USER')

| Test input                     | 1     | 2    | 3     | 4    | 5    | 6    | Covered   |
|--------------------------------|-------|------|-------|------|------|------|-----------|
| order_total < 40000: True      | X     | X    | O     | X    | O    | X    | O         |
| order_total < 40000: False     | O     | O    | X     | O    | X    | O    | O         |
| weight_kg <= 5: True           | X     | X    | X     | X    | O    | X    | O         |
| weight_kg <= 5: False          | O     | O    | O     | O    | X    | O    | O         |
| weight_kg <= 20: True          | X     | O    | X     | O    | X    | O    | O     

## Path Coverage-Guided Generation

In [6]:
BUDGET = 100

test_inputs = []

for i in range(BUDGET):
    path_tracker.reset()

    # Get coverage of currently selected test inputs
    for selected_test_input in test_inputs:
        path_tracker.run_test(*selected_test_input)
    recent_coverage, _, _ = path_tracker.calculate_coverage()
    
    # Generate a new random test input
    random_test_input = generate_random_test_input()

    # Run the new test input and check if coverage increases
    path_tracker.run_test(*random_test_input)
    new_coverage, _, _ = path_tracker.calculate_coverage()
    
    # Select the test input if it increases coverage
    if new_coverage > recent_coverage:
        test_inputs.append(random_test_input)

# Print final report for selected test inputs
path_tracker.reset()
for test_input in test_inputs:
    path_tracker.run_test(*test_input)
path_tracker.print_report()

PATH COVERAGE REPORT

Overall Coverage: 40.28% (58/144 items)
Total Tests: 58

Test Cases:
1: (81773, 13.59, 176, False, 'NONE', 'NEW_USER')
2: (106096, 17.46, 75, True, 'NONE', 'NEW_USER')
3: (94503, 39.48, 76, True, 'NONE', 'NONE')
4: (107717, 20.86, 146, True, 'WOW', 'NEW_USER')
5: (14980, 47.38, 89, True, 'WOW', 'NONE')
6: (23712, 24.48, 10, True, 'WOW', 'NONE')
7: (3265, 9.31, 84, True, 'NONE', 'NEW_USER')
8: (86738, 49.96, 38, True, 'NONE', 'NONE')
9: (32033, 22.77, 72, False, 'WOW', 'NEW_USER')
10: (17142, 36.66, 34, True, 'NONE', 'NONE')
11: (118450, 47.75, 111, True, 'WOW', 'NONE')
12: (113441, 47.82, 59, True, 'NONE', 'NEW_USER')
13: (108086, 26.52, 141, False, 'NONE', 'NONE')
14: (74832, 25.33, 29, False, 'NONE', 'NONE')
15: (116464, 36.54, 24, True, 'WOW', 'NEW_USER')
16: (24333, 2.15, 118, True, 'NONE', 'NEW_USER')
17: (83815, 27.56, 106, False, 'WOW', 'NONE')
18: (28087, 10.65, 76, True, 'WOW', 'NEW_USER')
19: (28732, 44.84, 126, False, 'NONE', 'NONE')
20: (27355, 13.19, 

## 🤔 Think: Smarter Random Input Generator

Our `pure` random input generator struggles to achieve high coverage.

**Why?** Uniform random inputs across six parameters often miss edge combinations that trigger meaningful branches (e.g., threshold boundaries, island surcharge, membership/coupon interactions).

---

### ✅ Your Task

Modify the `generate_random_test_input()` above to include **stronger bias toward boundary values** for:
- `order_total` around `40000`
- `weight_kg` around `5` and `20`
- `distance_km` around `10` and `50`
- categorical combinations of `membership` / `coupon_type`

This simple biasing strategy can significantly improve structural coverage.

---

## 🎯 Smarter Random Generators in Fuzzing

The ultimate goal of fuzzing is not just to generate random inputs —  
but to generate **smarter random inputs**.

Below are representative approaches used in practice:

| Approach                         | Required SuT Knowledge                     | Example                          |
|----------------------------------|--------------------------------------------|----------------------------------|
| Pure Random                      | 🟢 None (Black-box)                         | Uniform random sampling          |
| Biased Random                    | 🟢 Very Low (general heuristics only)       | Small-int bias, boundary values  |
| Mutation-Based                   | 🟡 Low (seed inputs required)               | AFL-style bit/byte mutations     |
| Evolutionary / Genetic           | 🟡 Grey-box (fitness such as coverage)      | Coverage-based genetic fuzzing   |
| Grammar-Based                    | 🟠 Input format / grammar knowledge         | JSON/XML grammar fuzzing         |
| Constraint-Based / Symbolic      | 🔴 High (White-box, path constraint access) | KLEE, symbolic execution engines |


---

### 🔎 Reflection

- Why do boundary-focused inputs improve coverage in this case?
- What kind of bias would help if the input domain were images? Strings? Network packets?
- How much knowledge about the system under test (SuT) should a fuzzing strategy assume?

Keep in mind:

> Pure random is rarely efficient in practice.  
> **The key to effective fuzzing is choosing (or designing) a strategy that fits your SuT.**